# Kuramoto Two-Oscillator Explorer
**BAMB! 2026 — Module 5**

$$
\dot{\theta}_1 = \omega_1 + \kappa\,\sin(\theta_2 - \theta_1) \\
\dot{\theta}_2 = \omega_2 + \kappa\,\sin(\theta_1 - \theta_2)
$$

Move the sliders and press **Run** to simulate. Each run starts fresh from the chosen initial phases.

| Scenario | Settings |
|---|---|
| Free rotation, no coupling | κ = 0 |
| Anti-phase lock (fly tripod) | κ < 0, f₁ = f₂ |
| In-phase lock | κ > 0, f₁ = f₂ |
| Phase-slipping | increase \|f₁−f₂\| until \|Δω\| > 2\|κ\| |
| Effect of noise | σ > 0 with strong coupling |

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display

%matplotlib inline
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#0d1117',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#8b949e',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'text.color':       '#c9d1d9',
    'grid.color':       '#21262d',
    'grid.linewidth':   0.8,
    'lines.linewidth':  2.0,
    'font.size':        11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

C1, C2, CGOLD, CGREEN = '#4a9eff', '#f87171', '#fbbf24', '#6ee7b7'

# --- Simulation ---
def simulate(f1, f2, kappa, sigma, ph1_0, ph2_0, T=8.0, dt=0.005):
    N  = int(T / dt)
    o1, o2 = 2*np.pi*f1, 2*np.pi*f2
    t  = np.arange(N) * dt
    T1, T2 = np.zeros(N), np.zeros(N)
    th1, th2 = ph1_0, ph2_0
    rng = np.random.default_rng(42)
    sq  = np.sqrt(dt)
    for i in range(N):
        T1[i], T2[i] = th1, th2
        n1, n2 = rng.standard_normal(2) * sigma * sq
        th1 += dt * (o1 + kappa * np.sin(th2 - th1)) + n1
        th2 += dt * (o2 + kappa * np.sin(th1 - th2)) + n2
    return t, T1, T2

def wrap(a):
    return (a + np.pi) % (2*np.pi) - np.pi

# --- Locking info ---
def locking_info(f1, f2, kappa):
    dw = 2*np.pi*(f1 - f2)
    bw = 2*abs(kappa)
    if abs(kappa) < 0.01:
        return 'Uncoupled  (κ ≈ 0): legs rotate independently', None
    if abs(dw) >= bw:
        return (f'Phase-slipping:  |Δω| = {abs(dw):.2f} rad/s  >  '
                f'locking BW 2|κ| = {bw:.2f} rad/s'), None
    sinFP = dw / (2*kappa)
    fp0   = np.arcsin(np.clip(sinFP, -1, 1))
    for fp in [fp0, np.pi - fp0]:
        fw = wrap(np.array([fp]))[0]
        if -2*kappa*np.cos(fw) < 0:
            label = ('anti-phase' if abs(abs(fw) - np.pi) < 0.3 else
                     'in-phase'   if abs(fw) < 0.3 else 'partial lock')
            return (f'Phase-locked  ({label})   Δφ* = {fw/np.pi:.2f}π   '
                    f'|Δω| = {abs(dw):.2f} < 2|κ| = {bw:.2f} rad/s'), fw
    return 'Phase-locked', None

# --- Phasor panel ---
def draw_phasor(ax, T, col, title, freq):
    theta = np.linspace(0, 2*np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), color=col, alpha=0.2, lw=1.5)
    ax.axhline(0, color='#30363d', lw=0.6)
    ax.axvline(0, color='#30363d', lw=0.6)
    # Trail
    trail_n = min(len(T), max(20, int(1.0 / (freq + 1e-9) / 0.005)))
    for i in range(max(0, len(T) - trail_n), len(T) - 1, 3):
        alpha = 0.05 + 0.3 * (i - (len(T) - trail_n)) / trail_n
        ax.plot([0, np.cos(T[i])], [0, np.sin(T[i])],
                color=col, alpha=alpha, lw=1.2)
    # Arrow
    th = T[-1]
    px, py = np.cos(th), np.sin(th)
    ax.annotate('', xy=(px, py), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5, mutation_scale=18))
    ax.plot(px, py, 'o', color=col, ms=8, zorder=5)
    # cos projection
    ax.plot([px, px], [py, 0], color=col, alpha=0.4, lw=1, ls='--')
    ax.plot(px, 0, 'o', color=col, ms=5, alpha=0.7)
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35)
    ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1])
    ax.tick_params(labelsize=8)
    ax.set_title(f'{title}\nθ = {(th % (2*np.pi))/np.pi:.2f}π', color=col, fontsize=10)

# --- Phase-difference phasor ---
def draw_phasor_diff(ax, T1, T2, col, fp_val):
    theta = np.linspace(0, 2*np.pi, 300)
    ax.plot(np.cos(theta), np.sin(theta), color='#30363d', lw=1.2)
    ax.axhline(0, color='#30363d', lw=0.6)
    ax.axvline(0, color='#30363d', lw=0.6)
    dphi_all = wrap(T1 - T2)
    trail_n  = min(len(dphi_all), 220)
    for i in range(max(0, len(dphi_all) - trail_n), len(dphi_all) - 1, 3):
        alpha = 0.05 + 0.35 * (i - (len(dphi_all) - trail_n)) / trail_n
        ax.plot([0, np.cos(dphi_all[i])], [0, np.sin(dphi_all[i])],
                color=col, alpha=alpha, lw=1.2)
    dp = dphi_all[-1]
    px, py = np.cos(dp), np.sin(dp)
    ax.annotate('', xy=(px, py), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5, mutation_scale=18))
    ax.plot(px, py, 'o', color=col, ms=8, zorder=5)
    if fp_val is not None:
        ax.plot(np.cos(fp_val), np.sin(fp_val), '*', color=CGREEN,
                ms=16, zorder=6, label=f'Δφ* = {fp_val/np.pi:.2f}π')
        ax.legend(loc='lower right', fontsize=8,
                  facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
    ax.set_xlim(-1.35, 1.35); ax.set_ylim(-1.35, 1.35)
    ax.set_xticks([-1, 0, 1]); ax.set_yticks([-1, 0, 1])
    ax.tick_params(labelsize=8)
    ax.set_title(f'Phase difference  Δφ\nΔφ = {dp/np.pi:.2f}π', color=col, fontsize=10)

# --- Main plot function ---
def run_and_plot(f1, f2, kappa, sigma, ph1_0_pi, ph2_0_pi):
    ph1_0 = ph1_0_pi * np.pi
    ph2_0 = ph2_0_pi * np.pi
    t, T1, T2 = simulate(f1, f2, kappa, sigma, ph1_0, ph2_0)
    dphi = wrap(T1 - T2)
    info_str, fp_val = locking_info(f1, f2, kappa)

    fig = plt.figure(figsize=(13, 8))
    gs  = gridspec.GridSpec(2, 3, figure=fig,
                            left=0.07, right=0.97,
                            top=0.88,  bottom=0.10,
                            wspace=0.38, hspace=0.55)

    ax1 = fig.add_subplot(gs[0, 0], aspect='equal')
    draw_phasor(ax1, T1, C1, 'Leg 1  (θ₁)', f1)

    ax2 = fig.add_subplot(gs[0, 1], aspect='equal')
    draw_phasor(ax2, T2, C2, 'Leg 2  (θ₂)', f2)

    ax3 = fig.add_subplot(gs[0, 2], aspect='equal')
    draw_phasor_diff(ax3, T1, T2, CGOLD, fp_val)

    ax4 = fig.add_subplot(gs[1, 0:2])
    ax4.plot(t, np.cos(T1), color=C1, label='Leg 1  cos(θ₁)', lw=1.8)
    ax4.plot(t, np.cos(T2), color=C2, label='Leg 2  cos(θ₂)', lw=1.8, alpha=0.85)
    ax4.axhline(0, color='#30363d', lw=0.8)
    ax4.set_xlabel('Time (s)')
    ax4.set_ylabel('cos(θ)')
    ax4.set_title('Leg signals', color='#c9d1d9', fontsize=11)
    ax4.legend(loc='upper right', fontsize=9,
               facecolor='#161b22', edgecolor='#30363d', labelcolor='#c9d1d9')
    ax4.set_ylim(-1.4, 1.4)
    ax4.grid(True)

    ax5 = fig.add_subplot(gs[1, 2])
    ax5.plot(t, dphi / np.pi, color=CGOLD, lw=1.8)
    ax5.axhline( 0, color='#30363d', lw=0.8)
    ax5.axhline( 1, color='#30363d', lw=0.8, ls='--')
    ax5.axhline(-1, color='#30363d', lw=0.8, ls='--')
    if fp_val is not None:
        ax5.axhline(fp_val/np.pi, color=CGREEN, lw=1.5, ls='--',
                    label=f'Δφ* = {fp_val/np.pi:.2f}π')
        ax5.legend(fontsize=9, facecolor='#161b22',
                   edgecolor='#30363d', labelcolor='#c9d1d9')
    ax5.set_xlabel('Time (s)')
    ax5.set_ylabel('Δφ / π')
    ax5.set_title('Phase difference  θ₁ − θ₂', color='#c9d1d9', fontsize=11)
    ax5.set_ylim(-1.3, 1.3)
    ax5.set_yticks([-1, -0.5, 0, 0.5, 1])
    ax5.set_yticklabels(['-π', '-π/2', '0', 'π/2', 'π'])
    ax5.grid(True)

    col = '#6ee7b7' if 'locked' in info_str else           '#fb923c' if 'slip'   in info_str else '#fbbf24'
    fig.suptitle(info_str, fontsize=11, color=col, y=0.97)
    plt.show()

print('Setup done — run the next cell to launch the widget.')

Setup done — run the next cell to launch the widget.


In [3]:
# --- Widgets ---
style  = {'description_width': '130px'}
layout = widgets.Layout(width='400px')

w_f1    = widgets.FloatSlider(value=2.0,  min=0.2, max=4.0,  step=0.05,
                               description='f₁  (Hz)',      style=style, layout=layout)
w_f2    = widgets.FloatSlider(value=2.0,  min=0.2, max=4.0,  step=0.05,
                               description='f₂  (Hz)',      style=style, layout=layout)
w_kappa = widgets.FloatSlider(value=-1.0, min=-3.0, max=3.0, step=0.05,
                               description='κ  (coupling)', style=style, layout=layout)
w_sigma = widgets.FloatSlider(value=0.0,  min=0.0,  max=3.0, step=0.05,
                               description='σ  (noise)',    style=style, layout=layout)
w_ph1   = widgets.FloatSlider(value=0.0,  min=0.0,  max=2.0, step=0.05,
                               description='θ₁(0) / π',    style=style, layout=layout)
w_ph2   = widgets.FloatSlider(value=1.0,  min=0.0,  max=2.0, step=0.05,
                               description='θ₂(0) / π',    style=style, layout=layout)

btn = widgets.Button(description='▶  Run simulation',
                     button_style='primary',
                     layout=widgets.Layout(width='200px', height='36px'))
out = widgets.Output()

def on_run(_):
    with out:
        out.clear_output(wait=True)
        run_and_plot(w_f1.value, w_f2.value, w_kappa.value,
                     w_sigma.value, w_ph1.value, w_ph2.value)

btn.on_click(on_run)

col1 = widgets.VBox([w_f1, w_f2, w_kappa],
                    layout=widgets.Layout(margin='0 24px 0 0'))
col2 = widgets.VBox([w_sigma, w_ph1, w_ph2])

display(widgets.VBox([
    widgets.HBox([col1, col2]),
    widgets.HBox([btn], layout=widgets.Layout(margin='12px 0 8px')),
    out
]))

on_run(None)  # run once on load